# 02 · Lightning 训练日志与重建检查

读取原生 Lightning `.ckpt` 和 CSV 日志，支持 MAE、CA-MAE、DiffMAE、CNN-MAE。
只做冻结模型评估，不启动训练。重建指标反映各自预训练目标，模型表征优劣以共同下游任务为准。

## 1. 路径与评估范围

将 `RUN_DIR` 指向训练输出目录，或设置 `PHENOSTATE_RUN_DIR`。
`best.ckpt` 按验证 loss 选择，`last.ckpt` 是最近验证结束的保存点。
`MAX_BATCHES=None` 评价完整 split；小整数用于预览。`NUM_WORKERS` 是读取线程数。

In [ ]:
import hashlib
import json
import os
from itertools import islice
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torchinfo import summary

from phenostate.data.inference_loader import local_well_batches
from phenostate.evaluate import evaluate_reconstruction
from phenostate.models import model_from_checkpoint

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').exists())
RUN_DIR = Path(os.environ.get('PHENOSTATE_RUN_DIR', PROJECT_ROOT / 'runs/comparison_patch8_50epochs/mae'))
CHECKPOINT = RUN_DIR / 'checkpoints/best.ckpt'
SPLIT = 'val'
MAX_BATCHES = None
NUM_WORKERS = 4
FORCE_FP32 = False
PREVIEW_WELLS = 1
RUN_GRAY_DIAGNOSTICS = False
GRAY_MAX_BATCHES = 32

## 2. Lightning CSV 日志

`train_loss_step` 是当前 microbatch 的重建 loss；`train_loss_epoch` 是整轮按样本数加权的均值。
`val_loss` 是完整验证集均值，训练结束才产生该记录。CSV 不同事件占不同行，空列为正常情况。
`step` 是 Lightning 的优化器步计数，梯度累积由 Trainer 处理。

In [ ]:
log_path = RUN_DIR / 'logs/metrics.csv'
logs = pd.read_csv(log_path) if log_path.exists() else pd.DataFrame()
if logs.empty:
    print('暂无 CSV 日志；已有 checkpoint 时可直接运行后面的单元格。')
else:
    display(logs.tail(10))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for column in ('train_loss_step', 'train_loss_epoch', 'val_loss'):
        if column in logs:
            values = logs[['step', column]].dropna()
            axes[0].plot(values['step'], values[column], label=column)
    for column in (c for c in logs if c.startswith('lr-')):
        values = logs[['step', column]].dropna()
        axes[1].plot(values['step'], values[column], label=column)
    axes[0].set(title='Reconstruction loss', ylabel='Masked MSE')
    axes[1].set(title='Learning rate', ylabel='LR')
    for ax in axes:
        ax.set_xlabel('Optimizer step')
        ax.grid(alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend()
    plt.tight_layout()
    plt.show()

## 3. 恢复模型与检查结构

模型结构取自 checkpoint 的 `hyper_parameters.model_config`，数据设置取自 `datamodule_hyper_parameters`。
可通过 `RXRX3_MDS_ROOT` 指定新的数据目录；其 manifest SHA 必须与训练时一致。
torchinfo 不传入图像，只展示结构与参数，避免摘要触发一次大规模前向。

In [ ]:
state = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
model = model_from_checkpoint(state)
data_settings = state['datamodule_hyper_parameters']
MDS_ROOT = Path(os.environ.get('RXRX3_MDS_ROOT', data_settings['mds_root']))
manifest_bytes = (MDS_ROOT / 'manifest.json').read_bytes()
manifest = json.loads(manifest_bytes)
if manifest.get('format') != 'rxrx3-core-well-mds-v1' or not manifest.get('complete'):
    raise ValueError('需要完整转换的 MDS')
if hashlib.sha256(manifest_bytes).hexdigest() != state['hyper_parameters']['manifest_sha256']:
    raise ValueError('MDS manifest 与训练时不一致')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device).eval()
BATCH_SIZE = data_settings['batch_size']
MASK_SEED = state['hyper_parameters']['seed'] + 10_000
USE_AMP = state['phenostate_precision'] == '16-mixed' and device.type == 'cuda' and not FORCE_FP32
print({'model': model.config.model_type, 'step': state['global_step'], 'device': str(device),
       'split': SPLIT, 'batch_size': BATCH_SIZE, 'amp': USE_AMP})
print(summary(model, depth=3, verbose=0))
del state

def new_eval_loader():
    return local_well_batches(MDS_ROOT, SPLIT, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                              expected_wells=manifest['counts'][SPLIT], pin_memory=device.type == 'cuda')

def hidden_patches(output):
    mask = output['mask'].bool()
    if mask.ndim == 2:
        mask = mask.unsqueeze(-1)
    return mask.expand_as(output['target'][..., 0])

## 4. 隐藏像素重建指标

Masked MSE/RMSE 越低越好，`reconstruction_skill = 1 - MSE / zero_baseline_MSE` 越高越好。
skill=0 表示与预测全零相同；它不是分类准确率。标准化目标中的零代表 patch/通道局部均值。
评估必须固定 split、batch size、seed、精度和目标空间；CA-MAE 掩码单位不同，DiffMAE 默认评价随机时间步去噪，不能仅凭重建 loss 排列模型优劣。

In [ ]:
with new_eval_loader() as loader:
    report = evaluate_reconstruction(model, loader, device, seed=MASK_SEED,
                                     amp=USE_AMP, max_batches=MAX_BATCHES)
expected = manifest['counts'][SPLIT]
report.update(complete_split=report['wells'] == expected, expected_wells=expected)
if MAX_BATCHES is None and not report['complete_split']:
    raise RuntimeError('评估数量不完整')
display(pd.Series({k: report[k] for k in ('masked_mse', 'masked_rmse', 'zero_baseline_mse',
        'reconstruction_skill', 'wells', 'complete_split', 'target_space', 'precision',
        'reconstruction_protocol')}, name='value').to_frame())
channel_df = pd.DataFrame({'channel': [f'ch{i+1}' for i in range(model.config.in_channels)],
                          'masked_mse': report['per_channel_mse'],
                          'zero_baseline_mse': report['per_channel_zero_baseline_mse'],
                          'reconstruction_skill': report['per_channel_reconstruction_skill']})
display(channel_df)
channel_df.plot.bar(x='channel', y=['masked_mse', 'zero_baseline_mse'], rot=0, figsize=(9, 4))
plt.ylabel('Masked MSE')
plt.tight_layout()
plt.show()

## 5. 六通道重建预览

每行一个通道：原图、可见输入、真实目标、隐藏位置用预测补全的目标、隐藏位置绝对误差。
目标和补全图使用同一色阶；标准化目标不等于原始 `[0,1]` 强度。DiffMAE 此处展示其随机时间步去噪输出。

In [ ]:
with new_eval_loader() as loader:
    batch = next(loader)
images = batch['image'].to(device)
with torch.inference_mode(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
    output = model(images, generator=torch.Generator(device=device).manual_seed(MASK_SEED))
    target = output['target']
    prediction = output['prediction'].float().reshape_as(target)
    target_images = model.unpatchify(target).cpu().numpy()
    prediction_images = model.unpatchify(prediction).cpu().numpy()
    pixel_mask = model.unpatchify(hidden_patches(output).unsqueeze(-1).expand_as(target)).cpu().numpy()
raw_images = images.cpu().numpy()
gray = plt.get_cmap('gray').copy()
gray.set_bad('0.5')
heat = plt.get_cmap('magma').copy()
heat.set_bad('0.5')
for well in range(min(PREVIEW_WELLS, len(raw_images))):
    fig, axes = plt.subplots(model.config.in_channels, 5, figsize=(16, 3 * model.config.in_channels), squeeze=False)
    for channel in range(model.config.in_channels):
        raw, truth, pred, hidden = (a[well, channel] for a in
                                    (raw_images, target_images, prediction_images, pixel_mask))
        hidden = hidden.astype(bool)
        lo, hi = np.quantile(truth, [0.01, 0.99])
        error = np.abs(pred - truth)
        axes[channel, 0].imshow(raw, cmap=gray, vmin=0, vmax=1)
        axes[channel, 1].imshow(np.ma.masked_where(hidden, raw), cmap=gray, vmin=0, vmax=1)
        axes[channel, 2].imshow(truth, cmap=gray, vmin=lo, vmax=max(hi, lo + 1e-6))
        axes[channel, 3].imshow(np.where(hidden, pred, truth), cmap=gray, vmin=lo, vmax=max(hi, lo + 1e-6))
        scale = max(float(np.quantile(error[hidden], 0.99)), 1e-6) if hidden.any() else 1e-6
        axes[channel, 4].imshow(np.ma.masked_where(~hidden, error), cmap=heat, vmin=0, vmax=scale)
        axes[channel, 0].set_ylabel(f'ch{channel+1}')
        for ax in axes[channel]:
            ax.set_xticks([])
            ax.set_yticks([])
    for ax, title in zip(axes[0], ['Original', 'Visible input', 'Target', 'Completed target', 'Hidden error']):
        ax.set_title(title)
    fig.suptitle(batch['well_id'][well])
    plt.tight_layout()
    plt.show()
del images, output, target, prediction

## 6. 可选灰度诊断

默认关闭。Gray Exact 为量化灰度完全匹配率，Nearest-5 为真值落入预测附近五个合法灰度档位的比例。
标准化目标反变换需要真实隐藏 patch 的均值和方差，因此这是条件诊断，会高估独立恢复原始强度的能力；不能作为分类 Top-5 准确率或正式主指标。

In [ ]:
@torch.inference_mode()
def gray_diagnostics(max_batches=32):
    totals = torch.zeros(model.config.in_channels, 3, dtype=torch.float64, device=device)
    with new_eval_loader() as loader:
        selected = loader if max_batches is None else islice(loader, max_batches)
        for index, batch in enumerate(selected):
            image = batch['image'].to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                output = model(image, generator=torch.Generator(device=device).manual_seed(MASK_SEED + index))
            raw = model.patchify(image).float()
            prediction = output['prediction'].float().reshape_as(raw)
            if model.config.norm_pix_loss:
                std = (raw.var(dim=-1, keepdim=True, unbiased=False) + 1e-6).sqrt()
                prediction = prediction * std + raw.mean(dim=-1, keepdim=True)
            truth = (raw * 255).round().clamp(0, 255).long()
            pred = (prediction * 255).round().clamp(0, 255).long()
            hidden = hidden_patches(output).unsqueeze(-1).expand_as(raw)
            lower = (pred - 2).clamp(0, 251)
            count = hidden.sum(dim=(0, 1, 3))
            exact = ((truth == pred) & hidden).sum(dim=(0, 1, 3))
            nearest = ((truth >= lower) & (truth <= lower + 4) & hidden).sum(dim=(0, 1, 3))
            totals += torch.stack([count, exact, nearest], dim=1)
    count, exact, nearest = totals.cpu().numpy().T
    return pd.DataFrame({'channel': [f'ch{i+1}' for i in range(len(count))],
                         'Gray_Exact_%': 100 * exact / np.maximum(count, 1),
                         'Gray_Nearest5_%': 100 * nearest / np.maximum(count, 1)})

if RUN_GRAY_DIAGNOSTICS:
    display(gray_diagnostics(GRAY_MAX_BATCHES))

所有重建分数均需注明 checkpoint、split、目标标准化方式、掩码协议、seed、batch size、精度和实际 well 数。
`best.ckpt` 只表示该模型自身验证重建目标最佳。正式表征比较请使用相同孔位、相同 gallery/query 和对照匹配规则的下游评价。